# Special topic: route a question across prose and tables

A corpus often holds two kinds of source: documents written in prose, and tables or a database
of figures. A question can need one, the other, or both. You can put everything in one index
and let retrieval sort it out, put a router in front of separate sources, or give an agent
tools for each source and let it choose. This notebook compares those designs on one corpus
and one question set, and reports accuracy, model calls, tokens, and routing errors per
question type.

The corpus is the 84 FinanceBench filings as `implementations/table_structured_reasoning` ingested them with Docling. Its
table-aware chunker splits each filing into prose chunks and table chunks, and the tables are
also loaded into SQLite, cell by cell, with each cell's row label and column header. The
question set has 20 questions of each type:

- `prose`: the answer is a phrase in the text and in no table of the filing;
- `table`: the answer is a table cell whose figure no prose chunk of the filing repeats;
- `both`: one question that asks for a table figure and a text statement from the same filing.

The arms differ only in who chooses the source:

| Arm | Who chooses | Sources it reads |
| --- | --- | --- |
| `U5`, `U10` | nobody | one dense index over prose and table chunks, top 5 or top 10 |
| `RE` | an embedding router: example questions per route, local `bge-small-en-v1.5` | the chosen route |
| `RL` | an LLM (large language model) router: route descriptions in a prompt, one call | the chosen route |
| `B` | nobody: every question goes to both sources | prose top 5 and SQL rows |
| `A` | the agent of `implementations/agent_native_retrieval`, on Google ADK, with the tools the FinanceBench profile generates | whatever it calls |
| `O` | the oracle: the gold label | the gold route |

The routes are `prose` (dense search over prose chunks, top 5, then one reading call),
`table` (one generated SQL query over the cell tables, one retry on an error or no rows, then
one reading call), and `both` (the same SQL rows and prose chunks, one reading call).

This is one run with one seed and no repeats, so only large differences are readable. The
reference ran on the on-prem `Qwen3_6-35B-A3B`, which also judged the answers. With
`RUN_EXPENSIVE = False` the tables below read that reference run.

In [1]:
import json
import sys
import warnings
from pathlib import Path

from aieng.advanced_rag import spend
from aieng.advanced_rag.models import choose_models


sys.path.insert(0, str(Path.cwd()))
import build_reference as br  # noqa: E402
import routing  # noqa: E402


RUN_EXPENSIVE = False
MODELS = choose_models(show=True)

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer); assuming $300.00 (set ADVANCED_RAG_KEY_BUDGET_USD to override)


## Prior art and why this design

Most arms follow a published design, and the code reuses what the project already has:

- One index over everything is the baseline: LangChain and LlamaIndex vector stores accept
  table text beside prose.
- The LLM router is LlamaIndex's `RouterQueryEngine` with an `LLMSingleSelector`: numbered
  choices with descriptions in a prompt and a JSON reply. LangChain's routing guide builds the
  same thing from a classification prompt and a branch. The `both` route corresponds to
  LlamaIndex's multi-selector and its `SQLAutoVectorQueryEngine`, which queries SQL and a
  vector index for one question.
- The embedding router is the algorithm of `semantic-router` (Aurelio AI, MIT): each route
  is a list of example utterances, and a question goes to the route whose closest examples
  score highest. It makes no model call.
- The table route is text-to-SQL in the shape of TAG (table-augmented generation; Biswal and others, "Text2SQL is Not
  Enough", 2024): synthesize a query, execute it, and generate the answer from the rows. BIRD
  and Spider are the text-to-SQL benchmarks behind it. The SQL runs on the guarded, read-only
  SQLite tables of the `agent_native_retrieval` FinanceBench profile.
- The agent is arm `P` of `agent_native_retrieval/describe_your_data.py` (the arm that gets the
  profile's generated tools): `sql`, `outline`, and `open_section` beside `search`, `grep`, and
  `open`. It runs on Google ADK (Agent Development Kit) through
  `aieng.advanced_rag.agents.session_loop`, so ADK tools, callbacks, and plugins can be added to it.
- Learned routers are left for a later block. RouteLLM learns to pick a strong or a weak model
  from preference data, Adaptive-RAG trains a small classifier on outcome labels to pick a
  retrieval strategy by query complexity, and StructRAG trains a router to pick a knowledge
  structure (table, graph, chunks). Each needs training labels. This notebook's per-route
  outcomes are the labels Adaptive-RAG uses (the cheapest route that answered), so they are a
  starting point.

None of LlamaIndex, LangChain, or `semantic-router` is in the project's lock file, so the
two routers are written out in `routing.py` in under 200 lines, on the project's cached and
metered model client and local embedder. Swapping in the libraries leaves the method
unchanged.

## The sources

`build_reference.py prepare` chunks every filing and embeds every chunk once (over an hour
on a shared 32-vCPU workstation). The tables are the three the FinanceBench profile declares: `filings`, `tables`,
and `cells`. The SQL source sees this schema, with the column descriptions the profile
gives:

In [2]:
tables = br.sql_tables()
print(tables.schema())

filings (84 rows): one row per filing
  doc_id text -- the filing's document, such as filings/3M_2018_10K.md
  filing text -- the filing name, company_period_type, such as 3M_2018_10K
  company text -- company name
  sector text -- GICS sector
  doc_type text -- 10k, 10q, 8k, or earnings
  period integer -- fiscal year the filing covers
  pages integer -- PDF pages
tables (7,600 rows): one row per table found in a filing
  table_id text -- filing#tNNN
  doc_id text -- the filing's document
  filing text -- the filing name
  page integer -- PDF page the table is on, matching the # Page N heading
  n_rows integer -- rows
  n_cols integer -- columns
  columns text -- the column headers joined by ' | ', often years
  first_row_label text -- the first row's label
cells (247,158 rows): one row per non-empty table cell outside the first column, with its row's label and its column's header; use LIKE on row_label to find a line item
  table_id text -- the table
  filing text -- the filing name


## The routers

The route descriptions are what the LLM router reads, and the example questions are what the
embedding router compares against. Both were written for the kind of question, before the
question set existed, and name no company in it.

In [3]:
for route in routing.FILING_ROUTES:
    print(f"{route.name}: {route.description}")
    print(f"  {len(route.utterances)} examples, such as: {route.utterances[0]}")
print()
print(routing.LLMRouter(routing.FILING_ROUTES, complete=None).prompt("<question>"))

prose: The text of the filings: business descriptions, risk factors, management's discussion, accounting policies, legal proceedings, and named people, places, and events. Use it for questions answered by a statement in words.
  8 examples, such as: What does the company say is the main risk to its supply chain?
table: The tables extracted from the filings, queried with SQL: every cell with its row label (a line item such as Total revenues) and its column header (often a year). Use it for questions that ask for a reported figure.
  8 examples, such as: What was the company's total revenue in fiscal 2021?
both: Both sources together. Use it for a question that asks for a reported figure and also for something the text states in words.
  6 examples, such as: What was the company's net income in 2020, and what did management cite as the main driver of the change?

Some choices are given below. Each is a source that can answer
questions about a collection of company filings. Pick the one c

The embedding router needs no model call, so you can try it on questions of your own here:

In [4]:
# Loading the embedder imports tqdm, which warns when ipywidgets is missing. The text
# progress bar works without it, and the warning would print a local install path.
warnings.filterwarnings("ignore", message="IProgress not found")
router = routing.EmbeddingRouter(routing.FILING_ROUTES, br.embedder())
for question in (
    "What was the company's free cash flow in 2021?",
    "What does the company say about its exposure to interest rate changes?",
    "What was goodwill at the end of 2020, and which acquisition does the filing say added it?",
):
    decision = router.route(question)
    scores = ", ".join(f"{k} {v:.3f}" for k, v in decision.scores.items())
    print(f"{decision.route:5}  {scores}  {question}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11412.21it/s]

table  prose 0.595, table 0.744, both 0.681  What was the company's free cash flow in 2021?
prose  prose 0.644, table 0.608, both 0.633  What does the company say about its exposure to interest rate changes?
both   prose 0.551, table 0.639, both 0.640  What was goodwill at the end of 2020, and which acquisition does the filing say added it?


## The question set

`build_reference.py questions` wrote and checked the questions on the on-prem models:
`Qwen3_6-35B-A3B` wrote each question from one table cell or one prose passage, and
`Qwen3_8-27B` had to answer it from that source alone, judged against the reference. A table
cell qualified only if its row label and column header name a single value in the filing, the
`cells` table holds it, and no prose chunk of the filing states the same figure. A prose
answer qualified only if it is copied from the passage and appears in no table chunk of the
filing. A `both` question merges one accepted table question and one accepted prose question
on the same filing. The question text stays in the local cache, and the committed summary holds
the counts and the funnel:

In [5]:
qset = json.loads((br.OUTPUTS / "question_set.json").read_text())
print(qset["counts"], "over", qset["filings"], "filings")
for row in qset["funnel"]:
    print(f"{row['kind']:6} {row['stage']:44} {row['passed']:4} of {row['entered']:4}")

{'prose': 20, 'table': 20, 'both': 20} over 31 filings
table  filing has a table-only cell                   35 of   36
prose  filing has running text                        36 of   36
table  question written                               70 of   70
table  question names the column's year               69 of   70
prose  question written                               72 of   72
prose  answer copied from the passage                 70 of   72
prose  answer absent from the filing's tables         59 of   70
table  checker answers from the source alone          68 of   69
prose  checker answers from the source alone          57 of   59
both   questions merged, years kept                   20 of   20


## The designs considered, and the ones run

Seven designs that differ in kind were listed before any run:

1. One index over everything (`U5`, `U10`): no routing.
2. A classifier router over separate sources, embedding-based (`RE`) or LLM-based (`RL`).
3. No router, every question to every source (`B`): routing as a cost saving only.
4. An agent with profile-generated tools that picks its own source (`A`).
5. The oracle (`O`): the ceiling for a router over these sources.
6. Decompose the question into one sub-question per source and merge the answers
   (LlamaIndex's `SubQuestionQueryEngine`).
7. One query language over both, SQL with a text-answer function (SUQL, Structured and
   Unstructured Query Language), so the router disappears into the query.

Designs 1 to 3 and 5 share one set of per-route answers: every route's answer is computed for every
question, and a router arm looks its decision up in them. That makes them cheap to run
together and lets the scoring say, for every routing mistake, whether the route chosen also
answered. Design 6 is recorded as a next experiment below. Design 7 was not run, because the project has
no engine for it.

## The arm table

With `RUN_EXPENSIVE = True` and no cached result of your own, the routes, the LLM router, and the agent
run on `MODELS.generation`, and `MODELS.judging` judges them. A cheap-tier model runs all 60
questions, and a mid or frontier model is capped at 4 per type. The reference arms used
about 1,400 model calls, over half of them the agent's. The next cell prints the estimate of rebuilding each
step on your model: about $4.20 in all on `gemini-3.5-flash-lite`, most of it the agent's. A live run needs the local chunks, embeddings, and question set, so run
`python build_reference.py prepare` and `python build_reference.py questions` first.

In [6]:
rows, outcome, origin = br.results(MODELS, live=RUN_EXPENSIVE)
print(origin)
print(f"{'arm':4} {'model':16} {'n':>3} {'acc':>6} {'calls':>6} {'in tok':>8} {'out tok':>7}  label")
for r in rows:
    print(
        f"{r['arm']:4} {r['model'][:16]:16} {r['questions']:3d} {r['accuracy']:6.3f} {r['calls']:6.2f} "
        f"{r['prompt_tokens']:8,.0f} {r['completion_tokens']:7,.0f}  {r['label']}"
    )

mixed-routing-pipelines reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                        452 calls, 567,304 prompt + 18,696 completion tokens
                        rebuilding with gemini-3.5-flash-lite: about 452 calls, est. $0.22 (set RUN_EXPENSIVE = True to rebuild)
mixed-routing-agent reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                    767 calls, 12,533,030 prompt + 65,117 completion tokens
                    rebuilding with gemini-3.5-flash-lite: about 767 calls, est. $3.92 (set RUN_EXPENSIVE = True to rebuild)
mixed-routing-judgements reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                         196 calls, 60,264 prompt + 9,462 completion tokens
                         rebuilding with gemini-3.8-flash: about 196 calls, est. $0.08 (set RUN_EXPENSIVE = True to rebuild)


from the step cache (reference, reference, reference)
arm  model              n    acc  calls   in tok out tok  label
U5   Qwen3_6-35B-A3B   60  0.467   1.00    1,630      13  one index over prose and tables, top 5
U10  Qwen3_6-35B-A3B   60  0.517   1.00    3,151      13  one index over prose and tables, top 10
RE   Qwen3_6-35B-A3B   60  0.083   2.50    1,822     224  embedding router (semantic-router style, local bge-small)
RL   Qwen3_6-35B-A3B   60  0.333   3.07    2,037     211  LLM router (LlamaIndex LLMSingleSelector style), one call
B    Qwen3_6-35B-A3B   60  0.350   2.53    2,684     233  no router: every question to both sources
A    Qwen3_6-35B-A3B   60  0.750  12.78  208,884   1,085  agent_native_retrieval agent with the FinanceBench profile's tools (arm P)
O    Qwen3_6-35B-A3B   60  0.333   2.00    1,754     165  oracle router: the gold label


`acc` is the share the judge marked correct, and `calls` and the token columns are per
question, the router's own call included. Per question type:

In [7]:
print(f"{'arm':4} " + "  ".join(f"{t:>21}" for t in br.QTYPES))
for r in rows:
    cells = "  ".join(
        f"{v['accuracy']:5.2f} {v['calls']:5.2f}c {v['prompt_tokens']:>7,.0f}t" for v in r["by_type"].values()
    )
    print(f"{r['arm']:4} {cells}")

arm                  prose                  table                   both
U5    0.60  1.00c   1,221t   0.60  1.00c   1,713t   0.20  1.00c   1,956t
U10   0.65  1.00c   2,616t   0.70  1.00c   3,295t   0.20  1.00c   3,542t
RE    0.10  2.50c   1,996t   0.15  2.70c   1,623t   0.00  2.30c   1,847t
RL    0.65  2.20c   1,289t   0.15  3.70c   1,835t   0.20  3.30c   2,987t
B     0.65  2.60c   2,830t   0.20  2.70c   2,491t   0.20  2.30c   2,732t
A     0.90  8.55c 105,638t   0.65 16.05c 272,178t   0.70 13.75c 248,835t
O     0.65  1.00c     987t   0.15  2.70c   1,542t   0.20  2.30c   2,732t


## Routing errors

For each arm that chooses, the confusion matrix of gold type against route taken. For the
agent, the route is the sources its tool calls touched: `sql` is the table source, and every
other tool reads text. A mismatch is *costly* when the route taken answered wrong and the gold
route answered right, and *rescued* the other way round.

In [8]:
for r in rows:
    report = r.get("routing")
    if not report:
        continue
    print(
        f"{r['arm']}: agreement {report['agreement']:.2f}, mismatches {report['mismatches']}, "
        f"costly {report['costly']}, rescued {report['rescued']}"
    )
    for gold, taken in report["confusion"].items():
        print(f"   gold {gold:5} -> " + ", ".join(f"{k} {v}" for k, v in sorted(taken.items())))

RE: agreement 0.35, mismatches 39, costly 15, rescued 0
   gold prose -> both 3, prose 1, table 16
   gold table -> both 2, table 18
   gold both  -> both 2, table 18
RL: agreement 0.93, mismatches 4, costly 0, rescued 0
   gold prose -> both 1, prose 17, table 2
   gold table -> both 1, table 19
   gold both  -> both 20
A: agreement 0.37, mismatches 38, costly 3, rescued 1
   gold prose -> both 4, prose 16
   gold table -> both 11, prose 9
   gold both  -> both 6, prose 14


How often each fixed route, the one-index arms, and the agent answered each question type, and
the hindsight label (the cheapest route that answered, or `None` when no route did):

In [9]:
for qtype, row in outcome.items():
    accs = ", ".join(f"{k} {v:.2f}" for k, v in row["route_accuracy"].items())
    print(f"{qtype:5} ({row['questions']}): {accs}")
    print(f"      hindsight {row['hindsight']}; answered by any arm: {row['any_route_correct']}")

prose (20): prose 0.65, table 0.00, both 0.65, U5 0.60, U10 0.65, A 0.90
      hindsight {'prose': 13, 'None': 7}; answered by any arm: 19
table (20): prose 0.05, table 0.15, both 0.20, U5 0.60, U10 0.70, A 0.65
      hindsight {'None': 16, 'table': 3, 'prose': 1}; answered by any arm: 18
both  (20): prose 0.05, table 0.00, both 0.20, U5 0.20, U10 0.20, A 0.70
      hindsight {'None': 16, 'both': 3, 'prose': 1}; answered by any arm: 15


## Follow-up: change the table source, keep the routers

The reference run pointed at the table source, so the follow-up changed only that source, in
two ways that differ in kind, and scored every router again on the same questions:

- `table_chunks`: dense search over the table chunks alone, top 5, with no SQL.
- `sql_linked`: the same SQL step, with the list of all 84 filings in the prompt and three
  rules drawn from the failed queries. A figure for one year can sit in the next year's
  filing. Match a line item with one distinctive word, because a segment or a period is often
  in the column label. Filter the period through the column label. These rules were written
  after reading six failed queries from this question set, so `sql_linked`'s table score is an
  in-sample estimate. The held-out section below tests it on 20 questions that no rule was drawn from.

Two more embedding routers ran with no model call: `REm` masks company names, years, and form
types in the questions and the examples, and `REl` uses the other 59 labeled questions as its
examples, leaving each question out of its own.

In [10]:
follow = br.followup_results()
print(
    f"{'source':13} {'arm':4} {'acc':>6} {'calls':>6} {'in tok':>7}  {'prose':>5} {'table':>5} {'both':>5}  agreement"
)
for source, block in follow["results"].items():
    for a in block["arms"]:
        types = " ".join(f"{v['accuracy']:5.2f}" for v in a["by_type"].values())
        agree = f"{a['routing']['agreement']:.2f}" if a["routing"] else ""
        print(
            f"{source:13} {a['arm']:4} {a['accuracy']:6.3f} {a['calls']:6.2f} {a['prompt_tokens']:7,.0f}  {types}  {agree}"
        )
print()
print("prose questions whose source chunk was retrieved:", follow["prose_gold_chunk_recall"])

source        arm     acc  calls  in tok  prose table  both  agreement
sql_plain     RE    0.083   2.50   1,822   0.10  0.15  0.00  0.35
sql_plain     REm   0.117   2.40   1,793   0.20  0.15  0.00  0.45
sql_plain     REl   0.333   2.40   2,263   0.65  0.20  0.15  0.38
sql_plain     RL    0.333   3.07   2,037   0.65  0.15  0.20  0.93
sql_plain     B     0.350   2.53   2,684   0.65  0.20  0.20  
sql_plain     O     0.333   2.00   1,754   0.65  0.15  0.20  
table_chunks  RE    0.267   1.00   2,065   0.10  0.60  0.10  0.35
table_chunks  REm   0.300   1.00   2,064   0.20  0.60  0.10  0.45
table_chunks  REl   0.483   1.00   2,479   0.65  0.55  0.25  0.38
table_chunks  RL    0.517   2.00   2,329   0.65  0.60  0.30  0.93
table_chunks  B     0.517   1.00   2,932   0.65  0.60  0.30  
table_chunks  O     0.517   1.00   1,970   0.65  0.60  0.30  
sql_linked    RE    0.300   2.20   4,693   0.10  0.70  0.10  0.35
sql_linked    REm   0.333   2.07   4,379   0.20  0.70  0.10  0.45
sql_linked    REl   0

## Held-out table questions

The `sql_linked` rules came from this question set's failed queries, so the follow-up cannot say
how they do on questions nobody read. `build_reference.py heldout` builds 20 held-out table questions
the way the reference table questions were built, with the same writer, checker, and filters, on
filings that no reference question uses. It then answers them with the same reader and judge from
four table sources:

- `sql_plain`: the reference run's SQL prompt.
- `sql_linked`: the follow-up's prompt, with the filing list and three matching rules.
- `table_chunks`: dense search over the table chunks, top 5, with no SQL. This is the control.
- `table_hybrid`: the rows of `sql_linked` and the top 5 table chunks in one reading call. It
  was designed after the first three scores were known, so its score is a first look, and the design has not been tested on fresh questions.

In [11]:
held = br.heldout_results()
produced = held["produced_by"]
print(f"{held['filings']} filings, none in the reference question set; answered and judged by {produced['model']}")
print(f"{'source':13} {'right':>5} {'no rows':>8} {'figure in rows':>15} {'calls':>6} {'in tok':>7}   reference set")
reference = {"sql_plain": "sql_plain", "sql_linked": "sql_linked", "table_chunks": "table_chunks"}
for name, entry in held["results"].items():
    if name == "paired":
        continue
    rows = f"{entry['sql_no_rows']:>8}" if "sql_no_rows" in entry else f"{'':>8}"
    found = f"{entry['sql_reference_in_rows']:>15}" if "sql_reference_in_rows" in entry else f"{'':>15}"
    before = ""
    if name in reference:
        accuracy = follow["results"][name]["route_accuracy"]["table"]["table"]
        before = f"{round(accuracy * 20)} of 20"
    print(
        f"{name:13} {entry['correct']:>5} {rows} {found} {entry['calls']:6.2f} {entry['prompt_tokens']:7,}   {before}"
    )
paired = held["results"]["paired"]
print(
    f"questions only sql_linked answered: {paired['linked_only']}; only sql_plain: {paired['plain_only']}; "
    f"answered by at least one of the three single sources: {paired['any_source']} of 20"
)

19 filings, none in the reference question set; answered and judged by Qwen3_6-35B-A3B
source        right  no rows  figure in rows  calls  in tok   reference set
sql_plain         7       11               7   2.75   1,649   3 of 20
sql_linked       12        3              13   2.25   4,508   14 of 20
table_chunks     15                            1.00   1,658   12 of 20
table_hybrid     16                            2.25   6,062   
questions only sql_linked answered: 6; only sql_plain: 1; answered by at least one of the three single sources: 19 of 20


One run on 20 questions, so read a difference of two or three questions as noise. `reference set`
is the same source on the 20 reference table questions, where the `sql_linked` rules were written.

- Table chunks answered 15 of 20, linked SQL 12, and the reference SQL prompt 7. The reference
  prompt returned no rows for 11 questions, against 13 in the reference set, so that failure
  repeats on filings it has not seen. The linked prompt returned no rows for 3.
- The linked prompt moved table questions from 7 to 12, a gain of 5, which is beyond noise. In the follow-up it moved
  them from 3 to 14. Six questions only it answered, and one only the reference prompt answered.
  The reference prompt scored higher on these questions than on the reference set (7 against 3), so
  the gain is 5 questions here against 11 there.
- Linked SQL and dense search are not separated by these questions: 14 against 12 in the
  reference set, 12 against 15 on the held-out set, and 26 against 27 over all 40.
- The linked queries returned the reference figure for 13 of 20 questions, and the reader answered
  12 of those 13 correctly. Most misses are queries that fetched the wrong rows.
- The two sources miss different questions. Table chunks answered 7 that linked SQL missed, and
  linked SQL answered 4 that table chunks missed. `table_hybrid` answered 16 of 20.

## What the reference run found

One run with no repeats, 20 questions per type, and `Qwen3_6-35B-A3B` answering and judging. Read
a difference of two or three questions as noise.

The oracle router, which always picks the right source, answered
20 of 60 questions. One index over everything, with no router, answered 31 at top 10. Knowing
the right source did not help because the table source failed. The generated SQL still
returned no rows after its retry for 13 of the 20 table questions, and the table route
answered 3 of the 20. The LLM router picked the gold route for 56 of 60 questions and made no
costly error, so it scored what the oracle scored. On these sources the table source set the
ceiling, and the router's classification reached it.

- The agent (`A`) answered 45 of 60, the most of any arm: 18 of 20 prose, 13 of 20 table, and
  14 of 20 two-source questions. It used 12.8 model calls and about 209,000 prompt tokens per
  question, against 1 call and about 3,200 tokens for `U10`, roughly 66 times the tokens. It
  called `sql` in 21 of its 60 runs, always beside text tools, `grep` was its most used tool,
  and it reached its 30-call budget on 8 questions.
- One index (`U10`) answered 14 of 20 table questions. The table-aware chunks keep a
  figure with its row label and column header, so dense search finds figures. It answered 4 of
  20 two-source questions, the same as every routed arm.
- The embedding router (`RE`) sent 52 of 60 questions to `table`, agreed with the gold label on
  21, and answered 5, with 15 costly errors. Every question names a company and a year, and so
  do the table examples, so similarity followed that surface and missed the information need.

The follow-up (above) changed the table source. Linked SQL answered 14 of 20 table questions
in-sample, table chunks 12, and plain SQL 3. On 20 held-out table questions (the section above),
table chunks answered 15, linked SQL 12, and plain SQL 7. With linked SQL the oracle, the LLM router, and sending
every question to both sources each answered 33 of 60. That is still 12 below the agent, and
two-source questions reached only 6 of 20 against the agent's 14. Masking entities raised the
embedding router's agreement from 21 to 27 of 60, and labeled neighbors gave 23. None came
near the LLM router's 56.

What the run supports, scoped to this corpus, these sources, and this model:

- Routing added no accuracy here. Sending every question to both sources (`B`)
  scored within one question of the oracle under all three table sources. Routing saved calls: under linked SQL the
  oracle used 1.7 calls per question and `B` 2.2.
- Two-source questions scored 4 of 20 with plain SQL and 6 of 20 with linked SQL under the
  oracle and under `B`, against 14 for the agent, so routing was not the limit. The `both`
  route searches once for the whole question, and the prose route's top 5 held the source
  passage for 11 of 20 prose questions, so one search likely misses one part. The agent
  searched again for each part.

Negative results, each with what would have to be true for the idea to win and the cheapest
test:

- The embedding router (`bge-small-en-v1.5`, six to eight hand-written examples per route, or
  the other labeled questions) agreed with the gold label on at most 27 of 60 questions,
  against 56 for the LLM router. It
  could win where the routes differ in vocabulary as well as intent, such as code against
  tickets, or with a classifier trained on a few hundred labeled questions, as Adaptive-RAG
  trains one. Cheapest test: a logistic regression on `bge-small` vectors of 100 synthetic
  questions per route, scored on these 60.
- Every routed design lost on two-source questions, 6 of 20 against 14 for the agent. It could
  win if each part of the question reached its own source. Cheapest test: one call splits the
  question into a table part and a text part, each part takes its own route, and one call
  merges the answers, as LlamaIndex's `SubQuestionQueryEngine` does. That is about 80 calls on
  the 20 two-source questions.
- Plain text-to-SQL answered 3 of 20 reference table questions and 7 of 20 held-out ones, and
  returned no rows for 13 and 11. Linked SQL answered 14 in-sample and 12 on held-out questions,
  so its gain over plain SQL is 5 questions out of sample, and dense search over table chunks
  answered 15. Its queries held the reference figure for 13 of the 20 held-out questions, so
  the misses are mostly queries that fetched the wrong rows. It could win if the generated
  query picked the right filing and line item more often. Cheapest test: the hybrid row, SQL
  rows read beside table chunks (16 of 20 on the held-out questions, designed after seeing them),
  on 20 more held-out table questions.
- The agent costs about 66 times the tokens of one index. It could lose its lead if a
  decomposing router reaches its two-source score at a few calls. The test is the
  decomposition test above.

## Route your own sources

The routers take any routes. To try them on your corpus:

1. Write one `routing.Route` per source, with a description of what the source holds and five
   to ten example questions in your users' words.
2. Put a callable behind each route that answers a question from that source. The
   `Sources` class in `build_reference.py` is one example, with dense search and guarded SQL.
3. Label 20 questions per route by hand, run every route on every question, and score each
   router with `routing.routing_report`. Count costly errors, not mismatches: a mismatch that
   still answers costs nothing.

If your tables are files, a data profile (`agent_native_retrieval`, `profiles/template.yaml`) gives the agent a
`sql` tool over them with no further code.

In [12]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
